# 19 — Effect of the prior choice (claim P1)
The production posterior assumes a uniform prior on the box α, β ∈ [0.4, 1.2]. This notebook measures how
much the results depend on that choice, and where.

**Priors compared**
| Label | Prior | How the posterior is obtained |
|---|---|---|
| U-box | uniform [0.4, 1.2]² (production) | production posterior (notebook 06) |
| U-narrow | uniform [0.6, 1.1]² — **excludes Sc5 (β = 0.50)** | retrained, 3 × 20k, training pairs = production-bank pairs inside the box (exact) |
| U-wide | uniform α ∈ [0.3, 1.3], β ∈ [0.4, 1.3] (largest box with a unique, unsaturated-low steady state; §1) | retrained, 3 × 20k, new 60k bank |
| H | truncated normal on the box, mean 1.0, sd 0.15 per axis ("faults are rare") | importance reweighting of U-box samples |
| D | truncated normal on the box, mean 0.7, sd 0.15 per axis ("degradation expected") | reweighting |
| LogU | log-uniform on the box, p ∝ 1/(αβ) (scale-invariant prior for multiplicative factors) | reweighting |

Reweighting is exact for priors on the same support (the posterior is ∝ likelihood × prior, the U-box
posterior ∝ likelihood); its quality is reported as the effective sample size (ESS).

**Questions**
1. Do the headline results (Sc1–Sc4) move when the prior changes? (data-dominated vs prior-dominated)
2. Where does the prior matter: near the thresholds (Sc8, Sc9), near the box edges (Sc5), and under a
   masked monitor (T-only features)?
3. What happens when the prior excludes the truth (U-narrow at Sc5), and does the raw NPE leakage outside
   the box flag it?
4. Calibration when the deployed population differs from the training prior: U-box posterior on truths drawn
   from H, vs the H-reweighted posterior.

**PRE-REGISTER (user to fix and date before running):**
* P1 "results are prior-robust" holds if, for Sc1–Sc4 and every prior whose support contains the truth:
  |posterior-mean shift| ≤ 0.5 U-box posterior sd (α and β), sd ratio within [0.8, 1.25], and the
  three-way decision unchanged in ≥ 95% of windows.
* Expected (reported, not gated): larger shifts at Sc8/Sc9; prior-dominated T-only posteriors;
  U-narrow coverage 0 at Sc5.
* ESS: a reweighted result is reported only where median ESS ≥ 200 of 2000 draws.
* Leakage detector for an excluding prior: flag if raw mass outside the box ≥ 0.2 (threshold fixed here).

Runtime: ~1–2 h (two retrained priors × 3 seeds; 60k new simulations). Needs notebook 06 and 02 (L-sets).

In [ ]:
import json, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "notebooks" and NB_DIR.parent.name == "system_III", f"run from system_III/notebooks (cwd {NB_DIR})"
ROOT = NB_DIR.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, system_iii
from system_iii import physics as ph, simulator as sim, features as fe, data as da, scenarios as scn
from system_iii import sensors as sen, information as info, config as cfg, rules, metrics as wm
for _m in (ph, sim, fe, da, scn, sen, info, cfg, rules, wm):
    importlib.reload(_m)
R, FIGS, DATA = ROOT / "results", ROOT / "figures", ROOT / "data"
for _d in (R, FIGS, DATA):
    _d.mkdir(exist_ok=True)
P = ph.PLANT
T0 = time.time()

In [ ]:
from system_iii import inference as inf
import pickle
PROD = json.loads((R / "06_production.json").read_text())
_m = [pickle.load(open(f, "rb")) for f in PROD["member_files"]]
post = inf.EnsemblePosterior(_m) if PROD["use_ensemble"] else _m[0]
print("production:", PROD["production_budget"], "ensemble" if PROD["use_ensemble"] else "single", PROD["feature_set"])

## 1. Check the wider box: unique steady state, no low-side saturation, cascade loop stable

In [ ]:
WIDE = (np.array([0.3, 0.4]), np.array([1.3, 1.3])); NARROW = (np.array([0.6, 0.6]), np.array([1.1, 1.1]))
bad = []
for a in np.linspace(WIDE[0][0], WIDE[1][0], 25):
    for b in np.linspace(WIDE[0][1], WIDE[1][1], 25):
        for dTi, dTci in [(0, 0), (2, 2), (-2, -2), (2, -2), (-2, 2)]:
            s = ph.ss_closed(P, a, b, Ti=P.Ti + dTi, Tci=P.Tci + dTci)
            if len(s) != 1 or s[0]["sat"] == -1:
                bad.append((a, b, dTi, dTci, len(s)))
print("wide box: grid points with multiple or low-saturated steady states:", len(bad))
assert not bad, "U-wide is not admissible; shrink it and log the change"
# controller check on the wide box (production control structure; see docs/realism_upgrade_2026-11-10.md)
sys.path.insert(0, str(ROOT / "design"))
if cfg.__dict__.get("CONTROL", "closed") == "cascade":
    from check_cascade import jac_eigs
    worst = max(1 / np.abs(ev.real).min() for a in np.linspace(0.3, 1.3, 11) for b in np.linspace(0.4, 1.3, 10)
                for ev in [jac_eigs(P, a, b, *cfg.CASCADE, tw=0.5, tf=0.1)[0]] if ev is not None)
    print("slowest closed-loop time constant on the wide box [min]:", round(worst, 1))

## 2. Helpers: box switching, priors, reweighting

In [ ]:
import contextlib, pickle
from scipy.stats import truncnorm
OUT = R / "19"; OUT.mkdir(exist_ok=True)
L = da.load_npz(DATA / "L_prod.npz")
SC = [k for k in scn.CLOSED]                                   # closed-loop catalogue scenarios

@contextlib.contextmanager
def box(lo, hi):
    """Temporarily set the prior box used by training, prior-truncated sampling and the metrics."""
    old = inf.LO.copy(), inf.HI.copy()
    inf.LO[:] = lo; inf.HI[:] = hi
    try:
        yield
    finally:
        inf.LO[:] = old[0]; inf.HI[:] = old[1]

def tn_logpdf(v, m, s, lo, hi):
    return truncnorm.logpdf(v, (lo - m) / s, (hi - m) / s, loc=m, scale=s)

LO0, HI0 = inf.LO.copy(), inf.HI.copy()
PRIORS = {   # log-density on the production box (constants irrelevant)
    "H": lambda th: tn_logpdf(th[..., 0], 1.0, 0.15, LO0[0], HI0[0]) + tn_logpdf(th[..., 1], 1.0, 0.15, LO0[1], HI0[1]),
    "D": lambda th: tn_logpdf(th[..., 0], 0.7, 0.15, LO0[0], HI0[0]) + tn_logpdf(th[..., 1], 0.7, 0.15, LO0[1], HI0[1]),
    "LogU": lambda th: -np.log(th[..., 0]) - np.log(th[..., 1]),
}

def reweight(samples, logp, n_out=1000, seed=0):
    """Sampling-importance-resampling of U-box posterior samples to prior exp(logp); returns (samples, ESS)."""
    rng = np.random.default_rng(seed); B, n, _ = samples.shape
    out = np.empty((B, n_out, 2), np.float32); ess = np.empty(B)
    lw = logp(samples.astype(float)); lw -= lw.max(1, keepdims=True); w = np.exp(lw); w /= w.sum(1, keepdims=True)
    for i in range(B):
        out[i] = samples[i, rng.choice(n, n_out, p=w[i])]; ess[i] = 1.0 / (w[i] ** 2).sum()
    return out, ess

def summarise(label, smp, theta, names, lo, hi, extra=None):
    _, rows = wm.evaluate_scenario_set(smp, theta, names, lo, hi)
    ta, tb = inf.three_way(smp)
    for r in rows:
        m = names == r["scenario"]
        r.update(prior=label, mean_a=float(smp[m, :, 0].mean()), mean_b=float(smp[m, :, 1].mean()),
                 tw_a=ta[m], tw_b=tb[m], **({k: float(np.median(v[m])) for k, v in (extra or {}).items()}))
    return rows

## 3. U-box (production) and the reweighted priors H, D, LogU — full features

In [ ]:
sel = np.isin(L["sc_name"], SC)
X, TH, NM = L["x"][sel], L["theta"][sel], L["sc_name"][sel]
base, leak = inf.sample_within_prior(post, X, 2000, rng_seed=0)
rows = summarise("U-box", base[:, :1000], TH, NM, LO0, HI0, {"leak": leak})
for name, lp in PRIORS.items():
    smp, ess = reweight(base, lp, seed=1)
    rows += summarise(name, smp, TH, NM, LO0, HI0, {"ess": ess})
full = pd.DataFrame(rows)

## 4. Retrained boxes: U-narrow (excludes Sc5) and U-wide

In [ ]:
bank = da.load_npz(DATA / "06_bank_160k.npz")
def train_box(tag, lo, hi, theta, x):
    ms = []
    with box(lo, hi):
        for s in range(3):
            g = OUT / f"post_{tag}_s{s}.pkl"
            if g.exists():
                ms.append(pickle.load(open(g, "rb"))); continue
            sl = slice(s * 20_000, (s + 1) * 20_000)
            p_, _ = inf.train_npe(theta[sl], x[sl], seed=s); pickle.dump(p_, open(g, "wb")); ms.append(p_)
    return inf.EnsemblePosterior(ms)
inside = np.all((bank["theta"] >= NARROW[0]) & (bank["theta"] <= NARROW[1]), 1)
print("bank pairs inside U-narrow:", inside.sum())
post_narrow = train_box("narrow", *NARROW, bank["theta"][inside], bank["x"][inside])
fw = DATA / "19_bank_wide_60k.npz"
if not fw.exists():
    thw = np.random.default_rng(19_000).uniform(WIDE[0], WIDE[1], size=(60_000, 2))
    da.save_npz(fw, da.make_windows(thw, 19_000, protocol=cfg.PROTOCOL, feature_set=cfg.FEATURE_SET))
bw = da.load_npz(fw)
post_wide = train_box("wide", *WIDE, bw["theta"], bw["x"])
for tag, p_, (lo, hi) in (("U-narrow", post_narrow, NARROW), ("U-wide", post_wide, WIDE)):
    with box(lo, hi):
        smp, lk = inf.sample_within_prior(p_, X, 1000, rng_seed=2)
        full = pd.concat([full, pd.DataFrame(summarise(tag, smp, TH, NM, lo, hi, {"leak": lk}))])
full.drop(columns=["tw_a", "tw_b"]).to_csv(R / "19_prior_full.csv", index=False)

## 5. Sensitivity table and the P1 test

In [ ]:
ref = full[full.prior == "U-box"].set_index("scenario")
def contains(prior, sc):
    a, b = scn.SCENARIOS[sc].alpha, scn.SCENARIOS[sc].beta
    lo, hi = {"U-narrow": NARROW, "U-wide": WIDE}.get(prior, (LO0, HI0))
    return lo[0] <= a <= hi[0] and lo[1] <= b <= hi[1]
out = []
for _, r in full[full.prior != "U-box"].iterrows():
    k = r.scenario; r0 = ref.loc[k]
    out.append(dict(prior=r.prior, scenario=k, contains_truth=contains(r.prior, k),
                    shift_a_sd=(r.mean_a - r0.mean_a) / r0.sd_a, shift_b_sd=(r.mean_b - r0.mean_b) / r0.sd_b,
                    sd_ratio_a=r.sd_a / r0.sd_a, sd_ratio_b=r.sd_b / r0.sd_b,
                    same_decision=float(np.mean((r.tw_a == r0.tw_a) & (r.tw_b == r0.tw_b))),
                    cov90_b=r.cov90_truth_b, class_acc=r.class_acc, ess=r.get("ess", np.nan), leak=r.get("leak", np.nan)))
sens = pd.DataFrame(out); sens.to_csv(R / "19_sensitivity.csv", index=False)
display(sens.pivot_table(index="scenario", columns="prior", values=["shift_b_sd", "shift_a_sd", "same_decision"]).round(2))
g = sens[sens.scenario.isin(scn.GATED) & sens.contains_truth & ~((sens.ess < 200) & sens.ess.notna())]
p1 = dict(max_abs_shift_sd=float(np.abs(g[["shift_a_sd", "shift_b_sd"]]).values.max()),
          sd_ratio_range=[float(g[["sd_ratio_a", "sd_ratio_b"]].values.min()), float(g[["sd_ratio_a", "sd_ratio_b"]].values.max())],
          min_same_decision=float(g.same_decision.min()))
p1["P1_holds"] = bool(p1["max_abs_shift_sd"] <= 0.5 and 0.8 <= p1["sd_ratio_range"][0] and p1["sd_ratio_range"][1] <= 1.25
                      and p1["min_same_decision"] >= 0.95)
print(p1); da.save_json(R / "19_P1.json", p1)

## 6. Prior excluding the truth (U-narrow at Sc5): does leakage flag it?

In [ ]:
n5 = full[(full.prior == "U-narrow")].set_index("scenario")
display(n5[["mean_b", "sd_b", "cov90_truth_b", "leak"]].round(3))
print("Sc5 flagged by leakage ≥ 0.2:", float(n5.loc["Sc5_severe_fouling", "leak"]) >= 0.2,
      "| largest leak among scenarios whose truth is inside:", float(n5.drop("Sc5_severe_fouling").leak.max()))

## 7. Masked monitor: the same priors on the T-only posterior (prior-dominated?)

In [ ]:
it = [fe.NAMES["S1m"].index(n) for n in fe.NAMES["T_only"]]
ms = []
for s in range(3):            # same members as notebook 08's masking arm (trained there if missing)
    g = R / "08" / f"post_M_Tonly_s{s}.pkl"; g.parent.mkdir(exist_ok=True)
    if g.exists():
        ms.append(pickle.load(open(g, "rb")))
    else:
        sl = slice(s * 40_000, (s + 1) * 40_000)
        p_, _ = inf.train_npe(bank["theta"][sl], bank["x"][sl][:, it], seed=s); pickle.dump(p_, open(g, "wb")); ms.append(p_)
post_T = inf.EnsemblePosterior(ms)
gs = np.isin(NM, scn.GATED + ["Sc5_severe_fouling"])
bT, _ = inf.sample_within_prior(post_T, X[gs][:, it], 2000, rng_seed=3)
prior_sd = (HI0 - LO0) / np.sqrt(12)
rowsT = []
for name, lp in [("U-box", None)] + list(PRIORS.items()):
    smp = bT[:, :1000] if lp is None else reweight(bT, lp, seed=4)[0]
    for k in np.unique(NM[gs]):
        m = NM[gs] == k
        rowsT.append(dict(features="T_only", prior=name, scenario=k, mean_b=float(smp[m, :, 1].mean()),
                          sd_b=float(smp[m, :, 1].std(1).mean()), contraction_b=1 - float(smp[m, :, 1].std(1).mean()) / prior_sd[1]))
for name in ["U-box"] + list(PRIORS):
    f_ = full[(full.prior == name) & full.scenario.isin(np.unique(NM[gs]))]
    rowsT += [dict(features="S1m", prior=name, scenario=r.scenario, mean_b=r.mean_b, sd_b=r.sd_b,
                   contraction_b=1 - r.sd_b / prior_sd[1]) for _, r in f_.iterrows()]
tT = pd.DataFrame(rowsT); tT.to_csv(R / "19_masked_vs_full.csv", index=False)
display(tT.pivot_table(index=["features", "scenario"], columns="prior", values="mean_b").round(3))
display(tT.pivot_table(index=["features", "scenario"], columns="prior", values="contraction_b").round(3))

## 8. Population different from the training prior (truths drawn from H)

In [ ]:
rng = np.random.default_rng(19_500)
thH = np.stack([truncnorm.rvs((LO0[j] - 1) / 0.15, (HI0[j] - 1) / 0.15, loc=1, scale=0.15, size=1000, random_state=rng)
                for j in (0, 1)], 1)
dH = da.make_windows(thH, 19_500, protocol=cfg.PROTOCOL, feature_set=cfg.FEATURE_SET)
sH, _ = inf.sample_within_prior(post, dH["x"], 2000, rng_seed=5)
pop = {"U-box posterior (mismatched prior)": sH[:, :1000], "H-reweighted posterior (matched)": reweight(sH, PRIORS["H"], seed=6)[0]}
rows = []
for k, smp in pop.items():
    lo90, hi90 = np.quantile(smp, 0.05, 1), np.quantile(smp, 0.95, 1)
    cov = ((dH["theta"] >= lo90) & (dH["theta"] <= hi90)).mean(0)
    rows.append(dict(posterior=k, cov90_a=cov[0], cov90_b=cov[1],
                     rmse_a=float(np.sqrt(((smp[:, :, 0].mean(1) - dH["theta"][:, 0]) ** 2).mean())),
                     rmse_b=float(np.sqrt(((smp[:, :, 1].mean(1) - dH["theta"][:, 1]) ** 2).mean()))))
popt = pd.DataFrame(rows); popt.to_csv(R / "19_population_mismatch.csv", index=False); display(popt.round(4))

## 9. Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True, sharey=True)
for ax, col in zip(axes, ["shift_a_sd", "shift_b_sd"]):
    pv = sens.pivot_table(index="scenario", columns="prior", values=col).reindex(SC)
    pv.plot.bar(ax=ax, width=0.8); ax.axhline(0.5, ls=":", c="k"); ax.axhline(-0.5, ls=":", c="k")
    ax.set_title(col.replace("_sd", "") + " (units of U-box posterior sd)"); ax.set_xlabel("")
fig.savefig(FIGS / "19_prior_shift.png", dpi=150); plt.show()
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
for (feat, k), d in tT.groupby(["features", "scenario"]):
    ax.plot(d.prior, d.mean_b, "o-" if feat == "S1m" else "s--", label=f"{feat} {k}")
ax.set_ylabel("posterior mean of β"); ax.legend(fontsize=7, ncol=2); ax.set_title("full features follow the data; T-only follows the prior")
fig.savefig(FIGS / "19_masked_prior_dominance.png", dpi=150); plt.show()

## Interpretation — to be written after the run
Write toward the storyline (`docs/system_III_plan.md` §1) and update the claims ledger (§2).